In [11]:
import glob
import os

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm
from scipy.ndimage import gaussian_filter1d
from scipy.stats import kendalltau, chi2

import utils
import paths as p


In [2]:
heatmap_dir = os.path.join(p.FIGURES_DIR, 'heatmaps')
os.makedirs(heatmap_dir, exist_ok=True)

activation_sequences_dir = os.path.join(heatmap_dir, "activation_sequences")
os.makedirs(activation_sequences_dir, exist_ok=True)

In [3]:
units_vetted = pd.read_csv(os.path.join(p.LOGS_DIR, 'units_vetted.csv'), index_col=0).sort_values('unit_id')
sessions_vetted = pd.read_csv(os.path.join(p.LOGS_DIR, 'sessions_vetted.csv'), index_col=0).sort_values('num_units', ascending=False)

In [4]:
units_qc_all = pd.read_csv(p.LOGS_DIR / "qc_per_unit_with_pass_flags.csv")
units_qc = units_qc_all[units_qc_all['qc_pass_all'] == True]
units_per_session = units_qc.groupby('session_id').size().reset_index(name='unit_count')

In [5]:
sessions_vetted = sessions_vetted.merge(
    units_per_session[['session_id', 'unit_count']],
    how='left',
    left_on='id',
    right_on='session_id'
).rename(columns={'unit_count': 'unit_qc_pass_count'})

In [6]:
sessions_filtered = sessions_vetted[sessions_vetted['unit_qc_pass_count'] >= 60]
len(sessions_filtered)

7

In [7]:
time_step = 0.1

In [8]:
def filter_units_by_presence(trials, units):
    n_trials = len(trials['trial_id'].unique())
    filtered_units = {uid: spikes_df for uid, spikes_df in units.items() if (spikes_df.groupby('trial_id').size() > 0).sum() / n_trials >= 0.8}
    return filtered_units

In [9]:
def compute_trial_activation(trial_spikes_df, trial_info, time_step, sigma=1):
    """
    Compute per-unit peak-time (sec) for one trial using argmax of smoothed binned rate.
    Returns a pd.Series indexed by unit_id; NaN for units with no spikes in this trial.
    """
    unit_ids = trial_spikes_df['unit_id'].unique()
    trial_len = float(trial_info['trial_length'])
    # edges from -dt .. trial_len+dt (matches your heatmap approach)
    bin_edges = np.arange(0 - time_step, trial_len + 2*time_step, time_step)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2.0

    peak_times = {}
    for unit_id in unit_ids:
        u = trial_spikes_df.loc[trial_spikes_df['unit_id'] == unit_id, 'trial_time'].to_numpy()
        if u.size == 0:
            peak_times[unit_id] = np.nan
            continue
        counts, _ = np.histogram(u, bins=bin_edges)
        rates = counts / time_step
        rates_smoothed = gaussian_filter1d(rates, sigma=sigma)
        peak_idx = int(np.argmax(rates_smoothed))
        peak_time = float(np.clip(bin_centers[peak_idx], 0.0, trial_len))
        peak_times[unit_id] = peak_time

    # Ensure all session units appear (even if absent in this trial)
    # We'll add missing unit_ids at the DF assembly step to keep the matrix rectangular.
    s = pd.Series(peak_times, name=int(trial_info['trial_id']))
    return s


def compute_session_activation_sequences(trials, units, time_step, sigma=1):
    """
    Build two matrices for a session:
        peak_time_df: rows=unit_id, cols=trial_id (sec)
        rank_df:      rows=unit_id, cols=trial_id (1=earliest; NaN = silent)
    """
    # Collect all unit_ids once from 'units' (preserve your mapping order)
    all_unit_ids = list(units.keys())
    all_trial_ids = trials['trial_id'].astype(int).tolist()

    # Build a long spikes df with unit_id, trial_id, trial_time
    spikes_list = []
    for unit_id, spike_df in units.items():
        if spike_df.empty:
            continue
        s = spike_df[['trial_id', 'trial_time']].copy()
        s['unit_id'] = unit_id
        spikes_list.append(s)

    if not spikes_list:
        # Empty session: return empty-shaped matrices
        peak_time_df = pd.DataFrame(index=all_unit_ids, columns=all_trial_ids, dtype=float)
        rank_df = peak_time_df.copy()
        return peak_time_df, rank_df

    all_spikes = pd.concat(spikes_list, ignore_index=True)

    # Compute per-trial peak times
    cols = []
    for _, trial_info in trials.iterrows():
        trial_id = int(trial_info['trial_id'])
        tri = all_spikes.loc[all_spikes['trial_id'] == trial_id]
        if tri.empty:
            # No spikes recorded for this trial across all units
            col = pd.Series({uid: np.nan for uid in all_unit_ids}, name=trial_id)
        else:
            col = compute_trial_activation(tri, trial_info, time_step, sigma=sigma)
        cols.append(col)

    peak_time_df = pd.concat(cols, axis=1)

    # Make sure every unit and trial is present (keep NaNs for silence)
    peak_time_df = peak_time_df.reindex(index=all_unit_ids)
    peak_time_df = peak_time_df.reindex(columns=all_trial_ids)

    # Rank within each trial (earlier peak time = smaller rank). NaNs stay NaN.
    rank_df = peak_time_df.rank(axis=0, method='first', na_option='keep', ascending=True)

    return peak_time_df, rank_df

In [12]:
# === Batch over your sessions (replacing your plotting loop) ===
regenerate = True
sessions_grouped = sessions_filtered.groupby("region")

with tqdm(total=len(sessions_filtered), desc="Computing activation sequences", unit="session") as pbar:
    for region, region_sessions in sessions_grouped:
        region_folder = os.path.join(activation_sequences_dir, "dfs",region)
        os.makedirs(region_folder, exist_ok=True)

        for _, session_info in region_sessions.iterrows():
            base = os.path.join(region_folder, f"{session_info['id']}")
            peak_time_path = f"{base}_peak_times.csv"
            rank_path = f"{base}_activation_ranks.csv"
            if os.path.exists(peak_time_path) and os.path.exists(rank_path) and not regenerate:
                pass  # File exists, you can choose to skip or overwrite
            else:
                _, trials, units = utils.get_session_data(session_info['id'])
                filtered_units = filter_units_by_presence(trials, units)
                peak_time_df, rank_df = compute_session_activation_sequences(trials, filtered_units, time_step, sigma=1)
                peak_time_df.to_csv(peak_time_path, index=True)
                rank_df.to_csv(rank_path, index=True)

            pbar.update(1)

Computing activation sequences: 100%|██████████| 7/7 [00:52<00:00,  7.50s/session]


In [ ]:
def plot_wait_length_histograms(session_id, trials, bins=100, save_path=None):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
    axes[0].hist(trials['wait_length'], bins=bins, color='gray', alpha=0.7)
    axes[0].set_title('All trials')
    axes[0].set_xlabel('Wait Length (s)')
    axes[0].set_ylabel('Count')
    axes[1].hist(trials.loc[trials['good'], 'wait_length'], bins=bins, color='green', alpha=0.7)
    axes[1].set_title('Good trials')
    axes[1].set_xlabel('Wait Length (s)')
    axes[2].hist(trials.loc[~trials['good'], 'wait_length'], bins=bins, color='red', alpha=0.7)
    axes[2].set_title('Not good trials')
    axes[2].set_xlabel('Wait Length (s)')
    fig.suptitle(session_id)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(os.path.join(save_path, f"{session_id}_tw_histo.png"))
        plt.close()
    else:
        plt.show()

In [ ]:
def per_neuron_reliability_stats(peak_time_df, trial_ids=None):
    """
    peak_time_df: rows=unit_id, cols=trial_id (seconds; NaN = silent)
    Returns a DataFrame with coverage, mean, std, cv, and normalized jitter.
    """
    df = peak_time_df if trial_ids is None else peak_time_df[trial_ids]
    coverage = df.notna().mean(axis=1)  # frac of trials with a peak
    mean_t = df.mean(axis=1, skipna=True)
    std_t = df.std(axis=1, ddof=1, skipna=True)
    cv_t = std_t / mean_t
    out = pd.DataFrame({
        'coverage': coverage,
        'mean_peak_s': mean_t,
        'std_peak_s': std_t,
        'cv_peak': cv_t
    })
    return out

In [ ]:
def plot_neuron_activation_time_stats(session_id, rel, save_path=None):
    plt.figure(figsize=(7,5))
    plt.scatter(rel['mean_peak_s'], rel['cv_peak'], alpha=0.7, s=8)
    plt.xlabel('Mean Peak Time (s)')
    plt.ylabel('Peak Time Coefficient of Variation')
    plt.title(session_id)
    if save_path is not None:
        plt.savefig(os.path.join(save_path, f"{session_id}_at_mean_vs_cv.png"))
        plt.close()
    else:
        plt.show()

In [ ]:
for region, region_sessions in sessions_grouped:
    region_folder = os.path.join(activation_sequences_dir, "plots", region)
    os.makedirs(region_folder, exist_ok=True)

    for _, session_info in region_sessions.iterrows():
        save_path = os.path.join(region_folder)

        session_id = session_info['id']
        _, trials, _ = utils.get_session_data(session_id)
        plot_wait_length_histograms(session_id,trials, bins=100, save_path=save_path)

        peak_time_df_path = os.path.join(activation_sequences_dir, 'dfs', region, f"{session_id}_peak_times.csv")
        peak_time_df = pd.read_csv(peak_time_df_path, index_col=0)

        trial_list_good = trials.loc[trials['good']==True, 'trial_id'].tolist()
        trial_list_good_str = [str(tid) for tid in trial_list_good] 

        try:
            rel = per_neuron_reliability_stats(peak_time_df, trial_ids=trial_list_good_str)
        except KeyError as e:
            print(f"Failed for session {session_id}: {e}")
            continue
        rel_path = os.path.join(activation_sequences_dir, 'dfs', region, f"{session_id}_peak_times_consistency.csv")
        rel.to_csv(rel_path, index=True)

        plot_neuron_activation_time_stats(session_id, rel, save_path=save_path)

In [ ]:
# session_info = sessions_filtered.iloc[4]
# session_id = session_info['id']
# print(session_id)

# events, trials, units = utils.get_session_data(session_id)
# peak_time_df_path = os.path.join(activation_sequences_dir, "dfs", session_info['region'], f"{session_id}_peak_times.csv")
# peak_time_df = pd.read_csv(peak_time_df_path, index_col=0)

# trial_list_good = trials.loc[trials['good']==True, 'trial_id'].tolist()
# trial_list_good_str = [str(tid) for tid in trial_list_good]

# rel = per_neuron_reliability_stats(peak_time_df, trial_ids=trial_list_good_str)
# plot_neuron_activation_time_stats(session_id, rel)


In [ ]:
for idx, session_info in sessions_filtered.iterrows():
    session_id = session_info['id']
    rel_path = os.path.join(activation_sequences_dir, 'dfs', session_info['region'], 
                            f"{session_id}_peak_times_consistency.csv")
    rel = pd.read_csv(rel_path, index_col=0)

    rel_high = rel[rel['cv_peak'] < 0.6]
    units_list = rel_high.index.tolist()
    if len(units_list) > 2:
        print(session_id)

In [ ]:
session_info = sessions_filtered[sessions_filtered.id=="RZ065_2025-02-22_str"].iloc[0]
session_id = session_info['id']
events, trials, units = utils.get_session_data(session_id)

trial_list_good = trials.loc[trials['good']==True, 'trial_id'].tolist()
trial_list_good_str = [str(tid) for tid in trial_list_good]

rel_path = os.path.join(activation_sequences_dir, 'dfs', session_info['region'], f"{session_id}_peak_times_consistency.csv")
rel = pd.read_csv(rel_path, index_col=0)

rel_high = rel[rel['cv_peak'] < 0.6].copy()

units_list = rel_high.index.tolist()
units_list

unit_id = units_list[4]
spikes = units[unit_id]

In [ ]:
unit_id

In [ ]:
peak_time = rel.loc[unit_id]['mean_peak_s']

In [ ]:
peak_time

In [ ]:
def plot_unit_trials_heatmap(
        spikes, trial_ids, time_step=0.05, sigma=1.0, 
        fig_path=None, normalize_row='none', common_length='max',
         sort_by='peak', vclip=(0, 98), event_markers=True):
    # Filter spikes to selected trials

    df = spikes[spikes['trial_id'].isin(trial_ids)].copy()

    # Determine trial duration
    if common_length == 'max':
        T = df[['trial_time', 'decision_time', 'cue_off_time']].max().max()
    else:
        T = float(common_length)

    # Bin edges and centers
    bin_edges = np.arange(-time_step, T + 2*time_step, time_step)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2.0

    # Compute smoothed firing rates for each trial
    heat = []
    for tid in trial_ids:
        tri = df[df['trial_id'] == tid]
        counts, _ = np.histogram(tri['trial_time'], bins=bin_edges)
        rates = gaussian_filter1d(counts / time_step, sigma=sigma)
        if normalize_row == 'max':
            rates = rates / (rates.max() or 1)
        heat.append(rates)
    heat = np.vstack(heat)

    # Sort trials by peak or event time
    if sort_by == 'peak':
        order = np.argsort(np.argmax(heat, axis=1))
    else:
        order = np.arange(len(trial_ids))
    heat_sorted = heat[order]
    trials_sorted = [trial_ids[i] for i in order]

    # Plot heatmap
    vmin, vmax = np.percentile(heat_sorted, vclip)
    fig, ax = plt.subplots(figsize=(12, 6))
    im = ax.imshow(heat_sorted, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
    ax.set_xlabel('Time (bins)')
    ax.set_ylabel('Trials')
    ax.set_xlim(0, 350)
    plt.colorbar(im, ax=ax, label='Smoothed rate')
    plt.tight_layout()
    if fig_path:
        plt.savefig(fig_path, dpi=250)
        plt.close(fig)
    else:
        plt.show()

    return {'trials_sorted': trials_sorted, 'bin_centers': bin_centers, 'heat_sorted': heat_sorted}

In [ ]:
result_dict = plot_unit_trials_heatmap(
    spikes=spikes,
    trial_ids=trial_list_good,
    time_step=0.05,
    sigma=1.0,
    fig_path=None,
    normalize_row='max',     # or 'session_fr' if your df has it; or 'none'
    common_length='max',     # or a fixed seconds, e.g. 12.0
    sort_by=None,          # or 'decision_time' / 'cue_on_time' / 'none'
    event_markers=True
)


In [ ]:
session_info = sessions_filtered[sessions_filtered.id=="RZ065_2025-02-22_str"].iloc[0]
session_id = session_info['id']
events, trials, units = utils.get_session_data(session_id)

trial_list_good = trials.loc[trials['good']==True, 'trial_id'].tolist()
trial_list_good_str = [str(tid) for tid in trial_list_good]

rel_path = os.path.join(activation_sequences_dir, 'dfs', session_info['region'], f"{session_id}_peak_times_consistency.csv")
rel = pd.read_csv(rel_path, index_col=0)

rel_high = rel[rel['cv_peak'] < 0.6].copy()

units_list = rel_high.index.tolist()
units_list

unit_id = units_list[4]
spikes = units[unit_id]

In [ ]:
df = spikes[spikes['trial_id'].isin(trial_list_good)].copy()

In [ ]:
T = trials[trials['trial_id'].isin(trial_list_good)].trial_length.max()

In [ ]:
sort_by = None
vclip=(0, 98)
time_step = 0.1

In [ ]:
# def get_x_max(heat_sorted):
#     # heat_sorted: shape (n_trials, n_bins)
#     n_trials = heat_sorted.shape[0]
#     nonzero_counts = (heat_sorted != 0).sum(axis=0)
#     threshold = int(0.8 * n_trials)
#     valid_bins = np.where(nonzero_counts >= threshold)[0]

#     if len(valid_bins) > 0:
#         return valid_bins[-1]  # rightmost bin index where >=80% trials are nonzero
#     else:
#         return(heat_sorted.shape[1]-1) 

In [ ]:
# Bin edges and centers
bin_edges = np.arange(-time_step, T + 2*time_step, time_step)

# Compute smoothed firing rates for each trial
heat = []
for tid in trial_list_good:
    tri = df[df['trial_id'] == tid]
    counts, _ = np.histogram(tri['trial_time'], bins=bin_edges)
    rates = gaussian_filter1d(counts / time_step, sigma=1)

    heat.append(rates)
heat = np.vstack(heat)

# Sort trials by deviation from peak_time
if sort_by == 'peak':
    trial_peak_times = [np.argmax(r) * time_step for r in heat]
    order = np.argsort(np.abs(np.array(trial_peak_times) - peak_time))
else:
    order = np.arange(len(trial_list_good))
heat_sorted = heat[order]
trials_sorted = [trial_list_good[i] for i in order]

# Plot heatmap
vmin, vmax = np.percentile(heat_sorted, vclip)
fig, ax = plt.subplots(figsize=(12, 6))
ax.axvline(x=peak_time/time_step)
im = ax.imshow(heat_sorted, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
ax.set_xlabel('Time (bins)')
ax.set_ylabel('Trials')
ax.set_xlim(0, 200)
plt.colorbar(im, ax=ax, label='Smoothed rate')
plt.tight_layout()

In [ ]:
heat_sorted

In [ ]:
# ---------- Kendall's W (with tie correction; handles NaNs via subsetting) ----------
def kendalls_w(rank_df, min_presence=0.8, tie_bottom_impute=False):
    """
    rank_df: rows=unit_id, cols=trial_id (smaller=earlier; NaN allowed)
    min_presence: keep neurons present in >= this fraction of trials
    tie_bottom_impute: if True, fill missing ranks in each trial with a tied bottom rank

    Returns: W, chi2_stat, p_value, n_items_used, m_judges
    """
    # keep trials with at least 2 non-NaN entries
    trial_mask = rank_df.notna().sum(axis=0) >= 2
    R = rank_df.loc[:, trial_mask].copy()
    if R.shape[1] < 2:
        return np.nan, np.nan, np.nan, 0, 0

    # filter neurons by presence
    presence = R.notna().mean(axis=1)
    R = R.loc[presence >= min_presence]
    if R.empty:
        return np.nan, np.nan, np.nan, 0, R.shape[1]

    # per-trial re-ranking (handle ties consistently); optionally impute NaNs at bottom
    def rerank_col(col):
        c = col.copy()
        if tie_bottom_impute and c.isna().any():
            # put missing as max+1 (all tied at bottom)
            max_observed = c.max(skipna=True)
            c = c.fillna(max_observed + 1)
        # convert times to ranks with ties averaged
        # if entries are already ranks, this preserves order; if times, this ranks them
        return c.rank(method='average', na_option='keep', ascending=True)

    Rr = R.apply(rerank_col, axis=0)

    # drop any rows still having NaN (if not imputing)
    if not tie_bottom_impute:
        Rr = Rr.dropna(axis=0, how='any')
    n, m = Rr.shape
    if n < 2 or m < 2:
        return np.nan, np.nan, np.nan, n, m

    # sums of ranks per item
    Ri = Rr.sum(axis=1)
    Rbar = Ri.mean()
    S = ((Ri - Rbar) ** 2).sum()

    # tie correction per judge (trial)
    T_sum = 0.0
    for col in Rr.columns:
        ranks = Rr[col].values
        # group by identical ranks
        _, counts = np.unique(ranks, return_counts=True)
        T_sum += np.sum(counts**3 - counts)

    denom = (m**2) * (n**3 - n) - m * T_sum
    if denom <= 0:
        return np.nan, np.nan, np.nan, n, m

    W = 12.0 * S / denom

    # chi-square approx for significance
    chi2_stat = m * (n - 1) * W
    p = 1.0 - chi2.cdf(chi2_stat, df=n - 1)
    return float(W), float(chi2_stat), float(p), int(n), int(m)

# ---------- Pairwise Kendall's tau between trials ----------
def pairwise_tau(rank_df, trial_ids=None, min_common=5):
    """
    Returns a DataFrame of tau-b between trials (using neurons observed in both).
    """
    cols = rank_df.columns if trial_ids is None else trial_ids
    cols = [c for c in cols if c in rank_df.columns]
    out = pd.DataFrame(index=cols, columns=cols, dtype=float)
    for i, c1 in enumerate(cols):
        r1 = rank_df[c1]
        for c2 in cols[i:]:
            r2 = rank_df[c2]
            common = r1.notna() & r2.notna()
            if common.sum() < min_common:
                tau = np.nan
            else:
                tau, _ = kendalltau(r1[common], r2[common], nan_policy='omit')
            out.loc[c1, c2] = tau
            out.loc[c2, c1] = tau
    return out

# ---------- Template order and distances ----------
def template_order(rank_df, trial_ids=None, method='median'):
    """
    Returns a 'template' rank vector across neurons (median or mean rank across trials).
    """
    R = rank_df if trial_ids is None else rank_df[trial_ids]
    if method == 'median':
        tpl = R.median(axis=1, skipna=True)
    else:
        tpl = R.mean(axis=1, skipna=True)
    # re-rank to get a clean total order (ties averaged)
    tpl_rank = tpl.rank(method='average', na_option='keep', ascending=True)
    return tpl_rank

def tau_to_template(rank_df, tpl_rank, min_common=5):
    """
    Kendall's tau of each trial's ranks vs the template ranks.
    """
    scores = {}
    for c in rank_df.columns:
        r = rank_df[c]
        common = r.notna() & tpl_rank.notna()
        if common.sum() < min_common:
            scores[c] = np.nan
        else:
            scores[c], _ = kendalltau(r[common], tpl_rank[common], nan_policy='omit')
    return pd.Series(scores, name='tau_to_template')


In [ ]:

# 2) Sequence/order consistency
W, chi2_stat, p, n_items, m_judges = kendalls_w(
    rank_df[trial_ids_similar], min_presence=0.8, tie_bottom_impute=False
)
print(f"Kendall's W={W:.3f} (n={n_items} neurons, m={m_judges} trials), p={p:.2e}")

tau_mat = pairwise_tau(rank_df, trial_ids=trial_ids_similar, min_common=5)
tpl = template_order(rank_df, trial_ids=trial_ids_similar, method='median')
tau_to_tpl = tau_to_template(rank_df[trial_ids_similar], tpl, min_common=5)